# Three composition anti-patterns

```yaml
title: "Three composition anti-patterns"
keywords: composition anti-pattern, over-chaining, shared skill that's really a tool, description collisions, skill vs tool, seam, handoff contract, dependency graph, JIT loading, list_skills, load_skill, FakeModel, Sonnet 4.6
estimated duration: 20
```

<a id="top"></a>

> **Lesson:** L23 — Skill patterns & composition. Reference lecture, the last of the four L23
> lectures; it lands **Objective 5** (recognize composition anti-patterns) and closes the arc that
> [L2304](L2304_lecture.ipynb) opened (the dependency graph and JIT-across-it).
> Read [L2301_intro.md](L2301_intro.md) first.
> **Roadmap:** [objectives.md](../../../../docs/origin/lesson_roadmaps/L23/objectives.md)
> **Anchor model: Claude Sonnet 4.6.**
>
> **Runs two ways.** By default this notebook drives the offline scripted `FakeModel` (no key,
> deterministic, restart-and-run-all passes). Set `ANTHROPIC_API_KEY` and the optional live cells
> drive real Sonnet 4.6.


## Contents

- [1. Setup](#1-setup)
- [2. Over-chaining](#2-over-chaining)
- [3. A shared "skill" that's really a tool](#3-a-shared-skill-thats-really-a-tool)
- [4. Description collisions (run it)](#4-description-collisions-run-it)
- [5. Tie back to the payoff](#5-tie-back-to-the-payoff)
- [6. Takeaways](#6-takeaways)


## 1. Setup

Composition is not free. Splitting one capability into several skills, wiring handoffs, and sharing
a lower-level skill all add moving parts — extra `load_skill` hops, extra model turns, extra
descriptions the model has to tell apart. That spend only pays off under **one condition**:

> A skill *system* is a win only when it is **cheaper to run and easier to reason about** than the
> monolith (one mega-skill, or one long prompt) it replaced.

If it fails that test, you have taken a thing that worked and made it slower, pricier, and harder to
debug — while feeling productive, because "more, smaller pieces" *looks* like good engineering. This
lecture names the **three ways composition silently gets worse**, and cures each:

1. **Over-chaining** — so many tiny handoffs that the pipeline is mostly overhead.
2. **A shared "skill" that's really a tool** — the wrong *container* for a deterministic op.
3. **Description collisions** — two skills the model can't tell apart, so selection degrades across
   the whole set.

Each is one of the good composition moves *inverted* — we'll name that mapping at the end. The setup
cell below is the shared L23 runtime from [L2304](L2304_lecture.ipynb): the `Skill` type, the
`REGISTRY` of real curriculum-authoring skills, the two JIT tools (`list_skills`, `load_skill`), and
the helpers `loaded_skills` / `dependency_edges`. It runs offline with no key.


In [1]:
from __future__ import annotations

import re
from collections.abc import Callable
from dataclasses import dataclass
from typing import Annotated

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage

from fluffy_potato_curriculum.common.config import get_settings
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)

SONNET = "claude-sonnet-4-6"
# Offline is the default. Set ANTHROPIC_API_KEY to light up the optional live cells.
LIVE = bool(get_settings().anthropic_api_key)


@dataclass(frozen=True)
class Skill:
    """A skill: a name, a description that says WHEN it applies, and a body."""

    name: str
    description: str
    body: str


# A tiny stand-in for this repo's real curriculum-authoring skill system. Bodies
# reference load_skill("X") so the dependency edges are literal, runnable text.
REGISTRY: dict[str, Skill] = {
    "author-lesson-roadmap": Skill(
        name="author-lesson-roadmap",
        description="Draft a lesson's roadmap docs (objectives, demos) from the PRD. Stage 1.",
        body=(
            "1. Read the PRD row and format specs.\n"
            "2. Draft objectives.md and demos_or_activities.md.\n"
            "3. Validate cross-references: load_skill('check-lesson-links').\n"
            "4. Hand off to stage 2: load_skill('generate-materials-from-roadmap')."
        ),
    ),
    "generate-materials-from-roadmap": Skill(
        name="generate-materials-from-roadmap",
        description="Generate student materials (lectures, labs) from a roadmap. Stage 2.",
        body=(
            "1. Read the roadmap docs produced by stage 1.\n"
            "2. Generate intro, lectures, and lab pairs.\n"
            "3. Validate cross-references: load_skill('check-lesson-links')."
        ),
    ),
    "check-lesson-links": Skill(
        name="check-lesson-links",
        description="Check one document's markdown links; judge each unresolved one. Shared.",
        body="Run extract_links.py, then classify each unresolved link (broken / placeholder).",
    ),
}


def make_skill_tools(registry: dict[str, Skill]) -> list[Callable[..., str]]:
    """Build the two JIT tools over a skill registry: discovery + load."""

    def list_skills() -> str:
        """Discovery: list every registered skill as `name: description`.

        Call this first to see what is available before choosing one to load.
        """
        return "\n".join(f"{s.name}: {s.description}" for s in registry.values())

    def load_skill(
        name: Annotated[str, "the exact skill name from list_skills to read into context"],
    ) -> str:
        """Load: read one skill's full body into context, or an error if unknown."""
        skill = registry.get(name)
        if skill is None:
            return f"no such skill: {name!r}"
        return skill.body

    return [list_skills, load_skill]


def loaded_skills(messages: list[AIMessage]) -> list[str]:
    """Which skills the agent actually pulled, in order, from the run history."""
    return [
        call["args"]["name"]
        for msg in messages
        if isinstance(msg, AIMessage)
        for call in msg.tool_calls
        if call["name"] == "load_skill"
    ]


def dependency_edges(registry: dict[str, Skill]) -> list[tuple[str, str]]:
    """Extract 'A -> B' edges: skill A's body that calls load_skill('B')."""
    pattern = re.compile(r"load_skill\(['\"]([^'\"]+)['\"]\)")
    edges: list[tuple[str, str]] = []
    for skill in registry.values():
        for target in pattern.findall(skill.body):
            edges.append((skill.name, target))
    return edges


print("setup ready — LIVE =", LIVE)

setup ready — LIVE = False


[↑ Back to top](#top)

## 2. Over-chaining

**Over-chaining** is decomposing a task into so many tiny handoffs that the pipeline is *mostly*
load-overhead — and every hop is one more place the model's `load_skill` selection can miss. The
symptom is a five-skill chain doing what a two-step skill would do: latency and miss-risk dominate
the actual work.

The tell is in the seams. Recall from [L2304](L2304_lecture.ipynb) (and Objective 3): a handoff
earns its keep only when the boundary is a **real, stable contract** — a well-defined artifact that
makes each half separately loadable, testable, and re-runnable. A handoff added *for tidiness*, with
a fuzzy seam, buys none of that. It is pure cost.

Below we build a toy five-skill linear chain — `s1 -> s2 -> s3 -> s4 -> s5`, each body just handing
off to the next — and read its dependency edges. Then we contrast a **two-step** version that does
the same work. `dependency_edges` counts the hops for us: an edge is literally one skill's body
calling `load_skill` on the next.


In [2]:
# A toy 5-skill linear chain: each step's body just hands off to the next.
# Nothing branches, nothing is reused — every seam is a handoff for tidiness only.
over_chained: dict[str, Skill] = {
    "s1": Skill("s1", "Step 1 of the widget pipeline.", "Do a little. Then load_skill('s2')."),
    "s2": Skill("s2", "Step 2 of the widget pipeline.", "Do a little. Then load_skill('s3')."),
    "s3": Skill("s3", "Step 3 of the widget pipeline.", "Do a little. Then load_skill('s4')."),
    "s4": Skill("s4", "Step 4 of the widget pipeline.", "Do a little. Then load_skill('s5')."),
    "s5": Skill("s5", "Step 5 of the widget pipeline.", "Do the last little bit. Done."),
}

# The same work, collapsed to a real 2-step seam (e.g. "prepare" -> "finish").
two_step: dict[str, Skill] = {
    "prepare": Skill(
        "prepare",
        "Prepare the widget: all upfront work, up to the one real reusable boundary.",
        "Do steps 1-4 in one skill. Then load_skill('finish').",
    ),
    "finish": Skill(
        "finish",
        "Finish the widget from a prepared input.",
        "Do the last step. Done.",
    ),
}

chain_hops = dependency_edges(over_chained)
step_hops = dependency_edges(two_step)

print("5-hop chain edges:", chain_hops)
print("  hops (load_skill calls on the path):", len(chain_hops))
print("2-step edges:     ", step_hops)
print("  hops:", len(step_hops))
print()
print(
    f"over-chaining spends {len(chain_hops)} loads (+ model turns) where {len(step_hops)} would do"
)

5-hop chain edges: [('s1', 's2'), ('s2', 's3'), ('s3', 's4'), ('s4', 's5')]
  hops (load_skill calls on the path): 4
2-step edges:      [('prepare', 'finish')]
  hops: 1

over-chaining spends 4 loads (+ model turns) where 1 would do


Side by side, the over-chained pipeline is four handoffs where the collapsed one is a single real
seam:

```mermaid
graph LR
    subgraph over-chained ["Over-chained: 4 hops, all overhead"]
        s1 --> s2 --> s3 --> s4 --> s5
    end
    subgraph collapsed ["Collapsed: 1 real seam"]
        prepare --> finish
    end
```

**Cure:** collapse hops. A handoff should exist because the seam is a **real reusable/testable
boundary** — a place another skill genuinely re-enters the pipeline, or a stage you want to run and
verify on its own — **not for tidiness**. If no one else consumes `s2`'s output and you never run
`s3` alone, `s1..s4` are one skill. Split on contracts, not on step numbers.

[↑ Back to top](#top)


## 3. A shared "skill" that's really a tool

The second anti-pattern hits the *other* good composition move — the **shared lower-level skill**
(the write-once-reuse-many node `C` in `A -> C <- B`). It goes wrong when that shared node turns out
not to need any judgment at all.

Apply L22's (and L08's) discriminator, the one intentional re-invocation of that line:

> **A tool is *called* with a schema; a skill is *read* as instructions.** Is the model **reading
> judgment** it has to apply, or **calling a deterministic op** with structured input and output?

If every caller invokes the shared "skill" the *same deterministic way* — same inputs, one right
answer, no taste involved — then it is a **tool wearing a skill's clothes**. The symptom is a
"skill" whose body is a one-line "run this op," with no criteria to weigh. Wrapping it in a
`SKILL.md` and making the model `load_skill` it is pure ceremony: you pay a load and a body to be
told to do the one thing the function already does.

Contrast this with the real shared skill in this repo,
[`example_skills/check_lesson_links`](example_skills/check_lesson_links/SKILL.md). It looks
mechanical, but it is a skill *done right*: the mechanical part (find the links, check which targets
exist) is delegated to a **script** (`extract_links.py`), and what's left in the body is
**judgment** — an unresolved link might be a real bug, a deliberate `L<NN>` forward-pointer, or a
sibling stage 2 hasn't generated yet. The model has to *decide* which. That judgment is why it earns
being a skill.

Below we build the anti-pattern's mirror image: a fake `add-two-numbers` "skill" whose body is a
deterministic op, and the same op as a plain function tool. They compute the identical thing — so
the skill wrapper is waste.


In [3]:
# A "skill" that is really a deterministic op: no judgment, one right answer.
fake_shared_skill = Skill(
    name="add-two-numbers",
    description="Shared helper: add two integers a and b. Called by several skills.",
    body="Return the integer sum of a and b. (That's the whole 'skill'.)",
)


# The same capability as what it actually is: a plain, called-with-a-schema tool.
def add_two_numbers(a: int, b: int) -> int:
    """Add two integers and return their sum."""
    return a + b


# Prove they're the same thing. The "skill" body is prose describing exactly what
# the one-line function does — so loading it as a skill buys nothing over calling
# the tool. There is no judgment for the model to read; there's an op to run.
print("skill body says:  ", fake_shared_skill.body)
print("tool computes:    ", add_two_numbers(2, 3))
print()
print("skill-vs-tool test (L22/L08): is the model READING judgment, or CALLING a deterministic op?")
print("  add-two-numbers -> deterministic op, structured I/O, no judgment -> it's a TOOL")
print("  check-lesson-links -> mechanical extraction scripted, JUDGMENT kept -> it's a SKILL")

skill body says:   Return the integer sum of a and b. (That's the whole 'skill'.)
tool computes:     5

skill-vs-tool test (L22/L08): is the model READING judgment, or CALLING a deterministic op?
  add-two-numbers -> deterministic op, structured I/O, no judgment -> it's a TOOL
  check-lesson-links -> mechanical extraction scripted, JUDGMENT kept -> it's a SKILL


**Cure:** make it a **tool** — a called-with-a-schema function like `add_two_numbers` — or, when a
script already does the work, *let the script be the interface* (the way `check-lesson-links`
delegates its mechanical half to `extract_links.py`). Drop the `SKILL.md` wrapper. Keep the skill
container only for the part that needs judgment; if nothing does, there is no skill to keep.

[↑ Back to top](#top)


## 4. Description collisions (run it)

The third anti-pattern attacks selection itself. In L22 you learned **the description is the
trigger** — the model picks a skill by matching its `description`. **Description collisions** are
that failure scaled to a *set*: when two skills' descriptions overlap, the model can't reliably tell
them apart, and selection degrades across the **whole** system, not just the colliding pair.

Here is a colliding pair — `refund-request` and `return-request` — with near-identical descriptions.
For a task that both plausibly match, the model has nothing to discriminate on. We script a
`FakeModel` run (offline, deterministic) where the model loads the **wrong** one, then read back
which skill it pulled with `loaded_skills`.


In [4]:
SYSTEM = (
    "You are a support agent. Call list_skills() to discover skills, then "
    "load_skill(name) to read the one you need and follow it."
)

# Two skills whose descriptions overlap so much the trigger can't discriminate.
colliding: dict[str, Skill] = {
    "refund-request": Skill(
        "refund-request",
        "Handle a customer who wants money back for an order.",  # <- collides
        "Look up the order, then issue a refund to the original payment method.",
    ),
    "return-request": Skill(
        "return-request",
        "Handle a customer who wants money back for an order.",  # <- identical trigger
        "Generate a prepaid return label; refund is issued once the item is received.",
    ),
}

# The customer already shipped the item back and wants their money — this is a RETURN.
# But the descriptions are identical, so a real model has nothing to go on. We script
# the plausible-wrong pick: it loads refund-request.
wrong_run = FakeModel(
    [
        tool_reply(tool_call("c1", "list_skills", {})),
        tool_reply(tool_call("c2", "load_skill", {"name": "refund-request"})),  # WRONG
        text_reply("Issued an immediate refund."),
    ]
)
agent = create_agent(wrong_run, make_skill_tools(colliding), system_prompt=SYSTEM)
task = "I already shipped the item back last week and I still haven't gotten my money."
messages = (await agent.ainvoke({"messages": [HumanMessage(content=task)]}))["messages"]

print("colliding descriptions:")
for s in colliding.values():
    print(f"  {s.name}: {s.description}")
print()
print(
    "skill the agent loaded:", loaded_skills(messages), "<- wrong: refunded without the return flow"
)

colliding descriptions:
  refund-request: Handle a customer who wants money back for an order.
  return-request: Handle a customer who wants money back for an order.

skill the agent loaded: ['refund-request'] <- wrong: refunded without the return flow


Now the cure: rewrite the two descriptions to be **mutually discriminating** — each says what it's
for *and implicitly what it's not*. `refund-request` is for when the item never shipped or is already
back; `return-request` is for when the customer still has the item and must send it first. With a
real trigger to match on, the same task now routes correctly. We re-run a scripted `FakeModel` that
loads the **right** one.


In [5]:
# Same two skills, descriptions rewritten to be mutually discriminating.
discriminating: dict[str, Skill] = {
    "refund-request": Skill(
        "refund-request",
        "Refund an order when there is nothing to send back "
        "(item never shipped, or already returned and received).",
        colliding["refund-request"].body,
    ),
    "return-request": Skill(
        "return-request",
        "Start a return when the customer still has the item and must ship it back "
        "before any money is issued.",
        colliding["return-request"].body,
    ),
}

# The customer says they already shipped it back -> return-request is the match now.
right_run = FakeModel(
    [
        tool_reply(tool_call("c1", "list_skills", {})),
        tool_reply(tool_call("c2", "load_skill", {"name": "return-request"})),  # RIGHT
        text_reply("Confirmed your return is in transit; refund issues on receipt."),
    ]
)
agent = create_agent(right_run, make_skill_tools(discriminating), system_prompt=SYSTEM)
messages = (await agent.ainvoke({"messages": [HumanMessage(content=task)]}))["messages"]

print("discriminating descriptions:")
for s in discriminating.values():
    print(f"  {s.name}: {s.description}")
print()
print("skill the agent loaded:", loaded_skills(messages), "<- right: return flow")

discriminating descriptions:
  refund-request: Refund an order when there is nothing to send back (item never shipped, or already returned and received).
  return-request: Start a return when the customer still has the item and must ship it back before any money is issued.

skill the agent loaded: ['return-request'] <- right: return flow


**Cure:** make descriptions **mutually discriminating** — each states what it's for *and implicitly
what it's not*, so no two triggers fire on the same task. This is a **system-level** design task, not
a per-skill afterthought: adding one skill can quietly collide with an existing one, so descriptions
have to stay distinct *across the whole set*.

> The scripted runs above are for reproducibility — we decided the pick in advance. On a *live*
> model the wrong pick is genuinely non-deterministic, and a **smaller / cheaper** model collides
> **sooner** than Sonnet 4.6: a bigger set and a weaker model both stress selection harder. If you
> have a key, the guarded cell below runs the colliding vs. discriminating registries against real
> Sonnet 4.6.

[↑ Back to top](#top)


In [6]:
# OPTIONAL live path — only runs if ANTHROPIC_API_KEY is set. Keyless runs skip it,
# so restart-and-run-all still passes offline. Selection is non-deterministic here.
if LIVE:
    from langchain_anthropic import ChatAnthropic

    real = ChatAnthropic(model=SONNET, max_tokens=200)
    for label, registry in [("colliding", colliding), ("discriminating", discriminating)]:
        agent = create_agent(real, make_skill_tools(registry), system_prompt=SYSTEM)
        messages = (await agent.ainvoke({"messages": [HumanMessage(content=task)]}))["messages"]
        print(f"{label:16s} -> live Sonnet loaded:", loaded_skills(messages) or "(none)")
else:
    print("LIVE is False — skipping the real-model run (set ANTHROPIC_API_KEY to enable).")

LIVE is False — skipping the real-model run (set ANTHROPIC_API_KEY to enable).


## 5. Tie back to the payoff

The three anti-patterns are not random hazards — each is one of L23's **good moves inverted**:

| Good move (L2304) | Inverted into the anti-pattern |
| --- | --- |
| **Sequential handoff** across a *real seam contract* | **Over-chaining** — handoffs with *no* real seam, split for tidiness |
| **Shared lower-level skill** — reuse in the right *container* | **A "skill" that's really a tool** — reuse in the *wrong* container |
| **Discriminating description** as the trigger (L22) | **Description collisions** — the trigger failure scaled to the whole set |

So the diagnostic is always the same question: *did this composition move make the system cheaper to
run and easier to reason about than the monolith it replaced?* Over-chaining fails the **cost** half
(more loads and turns for no reuse). The fake shared skill fails the **clarity** half (a load and a
body where a function call would be plainer). Collisions fail **both** (wasted selection turns *and*
the wrong skill runs). A skill system earns its complexity only when it beats the monolith on **cost
AND clarity** — otherwise the monolith was fine.

One model note carried over from Section 4: a **smaller / cheaper** model fails the collision
**sooner** than Sonnet 4.6. Robustness to these anti-patterns is not purely a design property — a
weaker model tolerates less overlap and fewer hops before it starts mis-selecting, so the same
skill set can be "fine" on Sonnet and broken on a cheaper model.

[↑ Back to top](#top)


## 6. Takeaways

- **Composition is only a win if the system beats the monolith on cost AND clarity.** More, smaller
  pieces can *look* like good engineering while making things slower, pricier, and harder to debug.
- **Over-chaining** — too many tiny handoffs; the pipeline is mostly load-overhead and every hop is
  a place selection can fail. *Cure:* collapse hops; split only on a **real reusable/testable seam**,
  not for tidiness.
- **A shared "skill" that's really a tool** — a deterministic op with structured I/O, wrapped in a
  `SKILL.md`. Apply L22/L08's line: is the model **reading judgment** or **calling a deterministic
  op**? *Cure:* make it a **tool** (or let the script be the interface) and drop the skill wrapper.
  Keep the skill only for the part that needs judgment — as `check-lesson-links` does.
- **Description collisions** — overlapping descriptions so the model can't reliably pick; selection
  degrades across the **whole set**. *Cure:* **mutually discriminating** descriptions (what it's for
  *and* implicitly what it's not) — a system-level task, not a per-skill afterthought.
- **Each anti-pattern is a good move inverted:** handoff without a real seam, reuse in the wrong
  container, description-as-trigger failure scaled to a set.
- A **smaller / cheaper** model fails these sooner than Sonnet 4.6 — robustness depends on the model,
  not just the design.

[↑ Back to top](#top)
